In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from mystatsfunctions import OLSE,LMoments
import sys
import pathlib
from fair import *
from fair import return_empty_emissions, run_FaIR
import argparse

from time import perf_counter

In [2]:
def interpolate_to_model_grid(ds, var):

    # load reference grid and pressure at model levels
    model_level_p = pd.read_csv('/home/e/ermis/Irene-damages/model_to_pressure_levels.csv')['ph [hPa]'].values[::-1][:137] # half levels pressure 
    ref_ds = xr.open_dataset(f'{AOPP_BASE_PATH}/postproc/deltas/aux/{var}_nc_ref.grb2')
    
    # horizontal interpolation
    tmp = ds.interp(latitude=ref_ds.latitude, longitude=ref_ds.longitude, method='linear')

    # vertical interpolation
    tmp = tmp.interp(level=model_level_p, method='linear', kwargs={'fill_value': 'extrapolate'})

    # add output into frame of reference dataset
    target = xr.zeros_like(ref_ds) + tmp.values[::-1]

    return target.fillna(0)

def get_HC5():

    ## HadCRUT5
    HC5 = xr.open_dataset('{}source/ancil/HadCRUT.5.0.2.0.analysis.summary_series.global.annual.nc'.format(AOPP_BASE_PATH))
    HC5 = HC5.tas_mean.to_pandas()
    HC5.index = HC5.index.year

    return HC5

def get_erf():

    ## ERF components from AR6
    erf_ar6 = pd.read_csv('{}source/ancil/AR6_ERF_1750-2019.csv'.format(AOPP_BASE_PATH),index_col=0)
    erf_ar6.loc[:,'ghg'] = erf_ar6.loc[:,'total_anthropogenic'] - erf_ar6.loc[:,'aerosol']
    
    ### extend ERF to 2022
    ssp245_erf = pd.read_csv('{}source/ancil/ERF_ssp245_1750-2500.csv'.format(AOPP_BASE_PATH),index_col=0)
    ssp245_erf['aerosol'] = ssp245_erf.loc[:,'aerosol-radiation_interactions'] + ssp245_erf.loc[:,'aerosol-cloud_interactions']
    ssp245_erf.loc[:,'ghg'] = ssp245_erf.loc[:,'total_anthropogenic'] - ssp245_erf.loc[:,'aerosol']
    
    for year in [2020, 2021, 2022]: # not using end_year here, why?
        erf_ar6.loc[year] = ssp245_erf.loc[year] * erf_ar6.loc[year-1] / ssp245_erf.loc[year-1]

    return erf_ar6

def get_AWI():

    end_year = 2022 # fixed by HC5
    HC5 = get_HC5()
    erf = get_erf()

    ## ant / nat FaIR run
    fair_erf = pd.DataFrame(index=erf.index,columns=pd.MultiIndex.from_product([['ant','aer','nat'],['forcing']]),
                            data=pd.concat([erf.loc[:,'ghg'],erf.loc[:,'aerosol'],
                                            erf.loc[:,'total_natural']], axis=1).values)
    fair_emms = return_empty_emissions(start_year=1750,end_year=end_year,scen_names=['ant','aer','nat'])
    fair_temps = run_FaIR(emissions_in=fair_emms,forcing_in=fair_erf)['T'].loc[1850:]

    ## regress HadCRUT5 onto FaIR temperature output & define anthropogenic warming index
    X = np.column_stack([np.ones(fair_temps.loc[:end_year].index.size),fair_temps.loc[:end_year]])
    Y = HC5.loc[1850:end_year].values[:,None]
    mlr = OLSE.multiple(Y)
    mlr.fit(X)
    AWI = ( mlr.B[1]*fair_temps.aer + mlr.B[2]*fair_temps.ant ).default

    # NOTE I could have one AWI that is longer that I use for the regression against ERA5 
    # and another that ends in 2022 for the regression against HC5. Need to implement later

    return AWI


In [3]:
# definitions
AOPP_BASE_PATH = '/gf5/predict/AWH019_ERMIS_ATMICP/DATA/'
DELTA_PATH = f'{AOPP_BASE_PATH}/postproc/deltas/'

PERTURB_MONTH = [6] # month to create delta for
VAR = 'd' # variable to create delta for

START_YEAR = 1979
END_YEAR = 2021

In [4]:
# # Load ERA5 data
# era5_var  = xr.open_dataset(f'{AOPP_BASE_PATH}/ERA5/{VAR}_monthly/{VAR}_mon_ERA5_0.25x0.25_197901-202512.nc').rename(
#     {'valid_time': 'time', 'pressure_level': 'level'}
# )
# sh_var = {'q': False, 't': True, 'd': True,
#             'vo': True, 'u': False, 'v': False}[VAR] # is this variable on the spherical harmonic grid?

# # Anthropogenic warming index
# awi = get_AWI()

# # Select only the specified month
# months = era5_var['time'].dt.month

# era5_var = era5_var.chunk({"time": -1}) # massive speedup!

# for month in PERTURB_MONTH:
    
#     var_years = era5_var.sel(time=months.isin([month])).groupby('time.year').mean(dim='time').sel(year=slice(START_YEAR, END_YEAR)) # select the specified month and average each year

#     print(f"### Regressing using data from {START_YEAR} to {END_YEAR} for month {month} ###")
    
#     # Interpolate variable
#     timeslices = [x for x in np.arange(START_YEAR,END_YEAR+1,1)] 
#     X = np.array([awi.loc[timeslice] for timeslice in timeslices])
#     X = X[:,None,None,None]
#     Y = var_years[VAR].squeeze().values
#     olsreg = OLSE.simple( Y = Y )

#     # create objects for computation
#     olsreg.X = np.ma.array(X, mask=olsreg._mask)

#     w = olsreg.W

#     x = olsreg.X
#     y = olsreg.Y
#     olsreg.fit( X = X )

#     # compute estimated attributable warming over 1850-1900 to endyear period
#     var3d_out = olsreg.b1 * (awi.loc[END_YEAR] - awi.loc[1850:1900].mean())

#     # create DataArray object
#     var3d_out = xr.zeros_like(var_years[VAR].isel(year=-1).squeeze()) + var3d_out
    
#     print("### Regridding and saving ###")
#     var3d_out.to_netcdf(f'{DELTA_PATH}/tmp_{month}_{VAR}.nc')

#     # Interpolate and save as nc file
#     # var_interp = interpolate_to_model_grid(var3d_out, VAR=VAR).to_netcdf(f'{DELTA_PATH}/{VAR}_{month}_delta_ERA5_{START_YEAR}-{END_YEAR}.nc')

In [5]:
# Interpolate and save as nc file
month = 6
var3d_out = xr.open_dataset(f'{DELTA_PATH}/tmp_{month}_{VAR}.nc')
# var_interp = interpolate_to_model_grid(var3d_out, var=VAR)#.to_netcdf(f'{DELTA_PATH}/{VAR}_{month}_delta_ERA5_{START_YEAR}-{END_YEAR}.nc')

In [12]:
var = VAR
ds = var3d_out[var]

model_level_p = pd.read_csv('/home/e/ermis/Irene-damages/model_to_pressure_levels.csv')['ph [hPa]'].values[::-1][:137] # half levels pressure 
ref_ds = xr.open_dataset(f'{AOPP_BASE_PATH}postproc/deltas/aux/{var}_reg_ref2.grb2')
# ref_ds = xr.open_dataset('/gf5/predict/AWH019_ERMIS_ATMICP/test_ATMICP/TEMPBLOB_IC/temperature_blob_model_grid.grb2')

# horizontal interpolation
tmp = ds.interp(latitude=ref_ds.latitude, longitude=ref_ds.longitude, method='linear')

# vertical interpolation
tmp = tmp.interp(level=model_level_p, method='linear', kwargs={'fill_value': 'extrapolate'})

# add output into frame of reference dataset
target = xr.zeros_like(ref_ds) + tmp.values[::-1]

In [13]:
target

<xarray.Dataset> Size: 4GB
Dimensions:     (hybrid: 137, latitude: 1280, longitude: 2576)
Coordinates:
    number      int64 8B 0
    time        datetime64[ns] 8B 2022-06-15
    step        timedelta64[ns] 8B 00:00:00
  * hybrid      (hybrid) float64 1kB 1.0 2.0 3.0 4.0 ... 134.0 135.0 136.0 137.0
  * latitude    (latitude) float64 10kB 89.89 89.75 89.61 ... -89.75 -89.89
  * longitude   (longitude) float64 21kB 0.0 0.1398 0.2795 ... 359.6 359.7 359.9
    valid_time  datetime64[ns] 8B 2022-06-15
Data variables:
    d           (hybrid, latitude, longitude) float64 4GB 3.432e-07 ... nan
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-06-17T15:07 GRIB to CDM+CF via cfgrib-0.9.1...

In [15]:
ref_ds = xr.open_dataset(f'{AOPP_BASE_PATH}/postproc/deltas/aux/q_reg_ref.grb2')
ref_ds

<xarray.Dataset> Size: 937MB
Dimensions:     (hybrid: 137, values: 1661440)
Coordinates:
    number      int64 8B ...
    time        datetime64[ns] 8B ...
    step        timedelta64[ns] 8B ...
  * hybrid      (hybrid) float64 1kB 1.0 2.0 3.0 4.0 ... 134.0 135.0 136.0 137.0
    latitude    (values) float64 13MB ...
    longitude   (values) float64 13MB ...
    valid_time  datetime64[ns] 8B ...
Dimensions without coordinates: values
Data variables:
    q           (hybrid, values) float32 910MB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-06-17T15:11 GRIB to CDM+CF via cfgrib-0.9.1...

In [16]:
ref_ds = xr.open_dataset(f'{AOPP_BASE_PATH}/postproc/deltas/aux/{var}_nc_ref2.grb2')
ref_ds

Ignoring index file '/gf5/predict/AWH019_ERMIS_ATMICP/DATA//postproc/deltas/aux/d_nc_ref2.grb2.5b7b6.idx' older than GRIB file


<xarray.Dataset> Size: 2GB
Dimensions:     (hybrid: 137, latitude: 1280, longitude: 2576)
Coordinates:
    number      int64 8B ...
    time        datetime64[ns] 8B ...
    step        timedelta64[ns] 8B ...
  * hybrid      (hybrid) float64 1kB 1.0 2.0 3.0 4.0 ... 134.0 135.0 136.0 137.0
  * latitude    (latitude) float64 10kB 89.89 89.75 89.61 ... -89.75 -89.89
  * longitude   (longitude) float64 21kB 0.0 0.1398 0.2795 ... 359.6 359.7 359.9
    valid_time  datetime64[ns] 8B ...
Data variables:
    d           (hybrid, latitude, longitude) float32 2GB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-06-17T11:15 GRIB to CDM+CF via cfgrib-0.9.1...

In [14]:
AOPP_BASE_PATH

'/gf5/predict/AWH019_ERMIS_ATMICP/DATA/'

In [15]:
ref_ds = xr.open_dataset(f'{AOPP_BASE_PATH}/postproc/deltas/aux/{var}_nc_ref.grb2')
ref_ds

Ignoring index file '/gf5/predict/AWH019_ERMIS_ATMICP/DATA//postproc/deltas/aux/d_nc_ref.grb2.5b7b6.idx' older than GRIB file


<xarray.Dataset> Size: 937MB
Dimensions:     (hybrid: 137, values: 1661440)
Coordinates:
    number      int64 8B ...
    time        datetime64[ns] 8B ...
    step        timedelta64[ns] 8B ...
  * hybrid      (hybrid) float64 1kB 1.0 2.0 3.0 4.0 ... 134.0 135.0 136.0 137.0
    latitude    (values) float64 13MB ...
    longitude   (values) float64 13MB ...
    valid_time  datetime64[ns] 8B ...
Dimensions without coordinates: values
Data variables:
    d           (hybrid, values) float32 910MB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-06-17T11:15 GRIB to CDM+CF via cfgrib-0.9.1...

In [ ]:
ref_ds = xr.open_dataset('/gf5/predict/AWH019_ERMIS_ATMICP/test_ATMICP/TEMPBLOB_IC/temperature_blob_model_grid.grb2')
ref_ds

<xarray.Dataset> Size: 2GB
Dimensions:     (hybrid: 137, latitude: 1280, longitude: 2560)
Coordinates:
    time        datetime64[ns] 8B ...
    step        timedelta64[ns] 8B ...
  * hybrid      (hybrid) float64 1kB 1.0 2.0 3.0 4.0 ... 134.0 135.0 136.0 137.0
  * latitude    (latitude) float64 10kB 89.89 89.75 89.61 ... -89.75 -89.89
  * longitude   (longitude) float64 20kB 0.0 0.1406 0.2812 ... 359.6 359.7 359.9
    valid_time  datetime64[ns] 8B ...
Data variables:
    t           (hybrid, latitude, longitude) float32 2GB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-06-16T15:01 GRIB to CDM+CF via cfgrib-0.9.1...

In [13]:
ref_ds = xr.open_dataset('/home/e/ermis/test_ATMICP/grid_q.nc', engine='netcdf4')
ref_ds

<xarray.Dataset> Size: 1GB
Dimensions:    (time: 1, longitude: 1440, latitude: 721, level: 137)
Coordinates:
  * time       (time) datetime64[ns] 8B 2023-10-11
  * longitude  (longitude) float32 6kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
  * latitude   (latitude) float32 3kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * level      (level) int32 548B 1 2 3 4 5 6 7 ... 131 132 133 134 135 136 137
Data variables:
    q          (time, level, latitude, longitude) float64 1GB ...
Attributes:
    CDI:          Climate Data Interface version 2.0.3 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Wed Aug 28 12:04:11 2024: cdo -f nc setgridtype,regular /ho...
    CDO:          Climate Data Operators version 2.0.3 (https://mpimet.mpg.de...